In [6]:
%load_ext autoreload
%autoreload 2

In [1]:
import os
import utils_z
from casebase import cityjson_parser as cjpar
import time

### 1. 处理 CityGML 数据并导入数据库

#### 1.1 测试：使用 citygml-tools 将1个 CityGML 转换为 CityJSON

In [12]:
citygml_tools_path = r"E:\0_mylib\citygml-tools-2.4.1\citygml-tools.bat"

utils_z.run_cmd(f'"{citygml_tools_path}" --version')

citygml-tools version 2.4.1
(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>




CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" --version', returncode=0, stdout='citygml-tools version 2.4.1\n(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>\n\n', stderr='')

In [5]:
test_xml_path = "E:\\2_data\\building_3d_opensource\\hamburg\\LoD1-DE_HH_2023-04-01\\LoD1_32_549_5936_1_HH.xml"

utils_z.run_cmd(f'"{citygml_tools_path}" to-cityjson {test_xml_path}')


[10:10:55 INFO] Starting citygml-tools.
[10:10:55 INFO] Executing 'to-cityjson' command.
[10:10:56 INFO] [1|1] Processing file E:\2_data\building_3d_opensource\hamburg\LoD1-DE_HH_2023-04-01\LoD1_32_549_5936_1_HH.xml.
[10:10:57 INFO] Writing output to file E:\2_data\building_3d_opensource\hamburg\LoD1-DE_HH_2023-04-01\LoD1_32_549_5936_1_HH.json.
[10:10:57 INFO] Total execution time: 01 s.
[10:10:57 INFO] citygml-tools successfully completed.



CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" to-cityjson E:\\2_data\\building_3d_opensource\\hamburg\\LoD1-DE_HH_2023-04-01\\LoD1_32_549_5936_1_HH.xml', returncode=0, stdout="[10:10:55 INFO] Starting citygml-tools.\n[10:10:55 INFO] Executing 'to-cityjson' command.\n[10:10:56 INFO] [1|1] Processing file E:\\2_data\\building_3d_opensource\\hamburg\\LoD1-DE_HH_2023-04-01\\LoD1_32_549_5936_1_HH.xml.\n[10:10:57 INFO] Writing output to file E:\\2_data\\building_3d_opensource\\hamburg\\LoD1-DE_HH_2023-04-01\\LoD1_32_549_5936_1_HH.json.\n[10:10:57 INFO] Total execution time: 01 s.\n[10:10:57 INFO] citygml-tools successfully completed.\n", stderr='')

#### 1.2 测试：解析并检查一个 CityJSON 数据

In [14]:
import json

with open("E:\\2_data\\building_3d_opensource\\hamburg\\LoD1-DE_HH_2023-04-01\\LoD1_32_549_5936_1_HH.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 查看顶层结构
print(data.keys())

# 查看第一栋建筑的属性
first_building_id = list(data["CityObjects"].keys())[0]
first_building = data["CityObjects"][first_building_id]
print(f"\n建筑ID：{first_building_id}")
print(f"类型：{first_building['type']}")
print(f"属性：{json.dumps(first_building.get('attributes', {}), indent=2, ensure_ascii=False)}")
print(f"几何类型：{[g['type'] for g in first_building.get('geometry', [])]}")

dict_keys(['type', 'version', 'CityObjects', 'transform', 'vertices', 'metadata'])

建筑ID：DEHHALKA10005SAX
类型：Building
属性：{
  "creationDate": "2023-02-23T00:00:00+08:00",
  "DatenquelleLage": "1000",
  "DatenquelleBodenhoehe": "1100",
  "Gemeindeschluessel": "02200227",
  "DatenquelleDachhoehe": "1000",
  "DatenquelleGeschossanzahl": "1000",
  "Geometrietyp2DReferenz": "3000",
  "Grundrissaktualitaet": "2023-01-26",
  "BezugspunktDach": "2000",
  "measuredHeight": 6.88,
  "function": "31001_1010",
  "storeysAboveGround": 2
}
几何类型：['Solid']


In [8]:
first_geom = first_building["geometry"][0]
print(f"几何类型：{first_geom['type']}")
print(f"lod：{first_geom.get('lod')}")

# 看看boundaries结构
boundaries = first_geom["boundaries"]
print(f"faces数量：{len(boundaries[0])}")
print(f"第一个face：{boundaries[0][0]}")

# 看看transform
print(f"\ntransform：{json.dumps(data['transform'], indent=2)}")
print(f"vertices总数：{len(data['vertices'])}")
print(f"前3个顶点：{data['vertices'][:3]}")

几何类型：Solid
lod：1
faces数量：8
第一个face：[[0, 1, 2, 3, 4, 5]]

transform：{
  "scale": [
    0.001,
    0.001,
    0.001
  ],
  "translate": [
    549013.226,
    5935992.289,
    11.652
  ]
}
vertices总数：16363
前3个顶点：[[361534, 965852, 3788], [373190, 966172, 3788], [373449, 956825, 3788]]


In [9]:
buildings = cjpar.parse_cityjson_lod1("E:\\2_data\\building_3d_opensource\\hamburg\\LoD1-DE_HH_2023-04-01\\LoD1_32_549_5936_1_HH.json")

print(f"解析出建筑数：{len(buildings)}")
b = buildings[0]
print(f"ID：{b['building_id']}")
print(f"高度：{b['height']}")
print(f"楼层：{b['floor_count']}")
print(f"功能：{b['function']}")
print(f"2D footprint：{b['geom_2d']}")

解析出建筑数：1110
ID：DEHHALKA10005SAX
高度：6.88
楼层：2
功能：31001_1010
2D footprint：POLYGON ((549374.76 5936958.141, 549386.416 5936958.461, 549386.675 5936949.114, 549377.3200000001 5936948.865999999, 549375.024 5936948.796, 549374.888 5936953.3889999995, 549374.76 5936958.141))


#### 1.3 创建建筑lod1数据库表并批量转换、导入数据

In [2]:
conn = utils_z.get_conn("Test20260413", "postgres", "we6666", "localhost", "5432")

In [ ]:
# xml批量转换为json
batch_json_output_dir = "E:\\2_data\\building_3d_opensource\\hamburg\\lod1_json"
batch_xml_input_dir = "E:\\2_data\\building_3d_opensource\\hamburg\\LoD1-DE_HH_2023-04-01"

xml_files = [f for f in os.listdir(batch_xml_input_dir) if f.endswith(".xml")]
print(f"共{len(xml_files)}个文件待转换")

errors = []
for i, filename in enumerate(xml_files):
    input_path = os.path.join(batch_xml_input_dir, filename)
    cmd = f'"{citygml_tools_path}" to-cityjson --output="{batch_json_output_dir}" "{input_path}"'
    try:
        utils_z.run_cmd(cmd)
        if (i + 1) % 50 == 0:
            print(f"进度：{i+1}/{len(xml_files)}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")

print(f"转换完成，失败{len(errors)}个")

In [3]:
# 创建LOD1建筑表
utils_z.run_sql("""
CREATE TABLE IF NOT EXISTS hamburg_buildings_lod1 (
    building_id     VARCHAR PRIMARY KEY,
    geom_2d         GEOMETRY(Polygon, 25832),
    height          FLOAT,
    floor_count     INTEGER,
    function        VARCHAR,
    roof_type       VARCHAR,
    year_built      INTEGER,
    geom_3d         GEOMETRY(PolyhedralSurfaceZ, 25832)
);
""", conn=conn)

utils_z.run_sql("CREATE INDEX IF NOT EXISTS buildings_lod1_geom_idx ON hamburg_buildings_lod1 USING GIST (geom_2d);", conn=conn)
print("LOD1建筑表创建完成")

LOD1建筑表创建完成


In [16]:
json_files = [f for f in os.listdir(batch_json_output_dir) if f.endswith(".json")]
print(f"共{len(json_files)}个文件待处理")

total = 0
errors = []

for i, filename in enumerate(json_files):
    filepath = os.path.join(batch_json_output_dir, filename)
    try:
        buildings = cjpar.parse_cityjson_lod1(filepath)
        print(f"\rProcessing {filename}: {len(buildings)} buildings", end='', flush=True)
        count = cjpar.insert_buildings_lod1(buildings, conn, lod1_table="hamburg_buildings_lod1")
        total += count
        if (i + 1) % 50 == 0:
            print()
            print(f"进度：{i+1}/{len(json_files)}，已入库建筑：{total}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")

print()  # To move to the next line after the loop
print(f"\n完成！共入库建筑：{total}")
print(f"失败文件数：{len(errors)}")

共784个文件待处理
Processing LoD1_32_554_5924_1_HH.json: 916 buildingss进度：50/784，已入库建筑：15328
Processing LoD1_32_557_5926_1_HH.json: 1033 buildings进度：100/784，已入库建筑：38867
Processing LoD1_32_559_5938_1_HH.json: 539 buildingss进度：150/784，已入库建筑：68189
Processing LoD1_32_561_5937_1_HH.json: 779 buildingss进度：200/784，已入库建筑：88981
Processing LoD1_32_563_5933_1_HH.json: 629 buildingss进度：250/784，已入库建筑：116545
Processing LoD1_32_565_5930_1_HH.json: 475 buildingss进度：300/784，已入库建筑：153498
Processing LoD1_32_567_5921_1_HH.json: 268 buildingss进度：350/784，已入库建筑：179724
Processing LoD1_32_568_5944_1_HH.json: 914 buildingss进度：400/784，已入库建筑：207030
Processing LoD1_32_570_5941_1_HH.json: 551 buildingss进度：450/784，已入库建筑：227768
Processing LoD1_32_572_5930_1_HH.json: 305 buildingss进度：500/784，已入库建筑：254366
Processing LoD1_32_573_5947_1_HH.json: 523 buildingss进度：550/784，已入库建筑：291165
Processing LoD1_32_575_5930_1_HH.json: 1 buildingssss进度：600/784，已入库建筑：314181
Processing LoD1_32_576_5945_1_HH.json: 632 buildingss进度：650/784，已入库建筑：

#### 1.4 更新block表

In [18]:
result = utils_z.run_sql("""
    SELECT constraint_name
    FROM information_schema.table_constraints
    WHERE table_name = 'hamburg_blocks'
    AND constraint_type = 'PRIMARY KEY';
""", conn=conn, fetch=True)

print(result)

[('blocks_pkey',)]


In [19]:
# 1. 添加新字段
utils_z.run_sql("""
    ALTER TABLE hamburg_blocks
    ADD COLUMN IF NOT EXISTS city VARCHAR,
    ADD COLUMN IF NOT EXISTS country VARCHAR,
    ADD COLUMN IF NOT EXISTS area_m2 FLOAT;
""", conn=conn)

# 2. 填入城市、国家、面积
utils_z.run_sql("""
    UPDATE hamburg_blocks
    SET city = 'Hamburg',
        country = 'Germany',
        area_m2 = ST_Area(geom);
""", conn=conn)

# 3. 删除主键约束
utils_z.run_sql("""
    ALTER TABLE hamburg_blocks
    DROP CONSTRAINT blocks_pkey;
""", conn=conn)

# 4. 把block_id从INTEGER改为VARCHAR，同时替换为新格式
utils_z.run_sql("""
    ALTER TABLE hamburg_blocks
    ALTER COLUMN block_id TYPE VARCHAR
    USING 'DE_HH_' || LPAD(block_id::TEXT, 6, '0');
""", conn=conn)

# 5. 重新设置主键
utils_z.run_sql("""
    ALTER TABLE hamburg_blocks
    ADD PRIMARY KEY (block_id);
""", conn=conn)

print("改造完成")

# 验证
result = utils_z.run_sql("""
    SELECT block_id, city, country, area_m2
    FROM hamburg_blocks
    LIMIT 5;
""", conn=conn, fetch=True)

for row in result:
    print(row)

改造完成
('DE_HH_000003', 'Hamburg', 'Germany', 146583.28380672264)
('DE_HH_000004', 'Hamburg', 'Germany', 68077.03149206015)
('DE_HH_000006', 'Hamburg', 'Germany', 42258.92792725274)
('DE_HH_000007', 'Hamburg', 'Germany', 64097.37637555693)
('DE_HH_000155', 'Hamburg', 'Germany', 18986.38463513086)


#### 1.5 街区与建筑空间叠合，记录建筑所属的街区

In [20]:
# 添加block_id字段
utils_z.run_sql("""
    ALTER TABLE hamburg_buildings_lod1
    ADD COLUMN IF NOT EXISTS block_id VARCHAR;
""", conn=conn)

# 建立空间索引
utils_z.run_sql("""
    CREATE INDEX IF NOT EXISTS buildings_lod1_geom_idx
    ON hamburg_buildings_lod1 USING GIST (geom_2d);
""", conn=conn)

print("字段和索引准备完成，开始空间叠合...")

字段和索引准备完成，开始空间叠合...


In [22]:
# 用重心落点规则批量判定
start_time = time.time()

utils_z.run_sql("""
    UPDATE hamburg_buildings_lod1 b
    SET block_id = (
        SELECT bl.block_id
        FROM hamburg_blocks bl
        WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
        LIMIT 1
    );
""", conn=conn)

end_time = time.time()
elapsed_time = end_time - start_time
print(f"空间叠合完成，耗时：{elapsed_time:.2f}秒")


空间叠合完成，耗时：23.50秒


In [23]:
# 检查数据结果
result = utils_z.run_sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(block_id) AS matched,
        COUNT(*) FILTER (WHERE block_id IS NULL) AS unmatched
    FROM hamburg_buildings_lod1;
""", conn=conn, fetch=True)

print(f"总建筑数：{result[0][0]}")
print(f"成功匹配block：{result[0][1]}")
print(f"未匹配block：{result[0][2]}")

总建筑数：384629
成功匹配block：354272
未匹配block：30357


#### 1.6 绘制检查

In [3]:
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
import py5
import random

In [4]:
# 随机选取100个有建筑的block
sql_blocks = """
    SELECT block_id, geom
    FROM hamburg_blocks
    WHERE block_id IN (
        SELECT DISTINCT block_id
        FROM hamburg_buildings_lod1
        WHERE block_id IS NOT NULL
    )
    ORDER BY RANDOM()
    LIMIT 50;
"""
gdf_blocks = gpd.read_postgis(sql_blocks, conn, geom_col="geom", crs="EPSG:25832")
blocks = list(gdf_blocks.itertuples(index=False, name='Block'))
print(f"随机选取了 {len(blocks)} 个block")

# 预加载所有建筑数据
buildings_data = {}
for block in blocks:
    block_id = block.block_id
    sql_buildings = f"""
        SELECT building_id, geom_2d
        FROM hamburg_buildings_lod1
        WHERE block_id = '{block_id}';
    """
    gdf_buildings = gpd.read_postgis(sql_buildings, conn, geom_col="geom_2d", crs="EPSG:25832")
    buildings_data[block_id] = list(gdf_buildings.geometry)

# 全局变量
current_index = 0

def all_polygons(g):
    if isinstance(g, Polygon):
        return [g]
    elif isinstance(g, MultiPolygon):
        return list(g.geoms)
    return []

def draw_polygon(poly, is_building=False):
    """绘制单个polygon，根据是否是建筑设置不同颜色"""
    coords = list(poly.exterior.coords)
    if is_building:
        py5.fill(180, 100, 100)  # 建筑：红棕色
        py5.stroke(120, 60, 60)
    else:
        py5.fill(220, 220, 210)  # block：浅灰色
        py5.stroke(80, 80, 80)

    py5.begin_shape()
    for x, y in coords:
        py5.vertex(x, y)
    py5.end_shape(py5.CLOSE)

def draw_current_block():
    global current_index
    if current_index < 0 or current_index >= len(blocks):
        return

    block = blocks[current_index]
    block_id = block.block_id
    block_geom = block.geom

    buildings_geoms = buildings_data[block_id]

    py5.background(240)

    W, H = py5.width, py5.height
    minx, miny, maxx, maxy = block_geom.bounds
    cw = maxx - minx
    ch = maxy - miny
    margin = 0.15
    scale = min((W * (1 - margin)) / cw, (H * (1 - margin)) / ch)
    cx = (minx + maxx) / 2
    cy = (miny + maxy) / 2

    py5.push_matrix()
    py5.translate(W / 2, H / 2)
    py5.scale(scale, -scale)
    py5.translate(-cx, -cy)

    # 先画block轮廓
    py5.stroke_weight(1 / scale)
    for p in all_polygons(block_geom):
        draw_polygon(p, is_building=False)

    # 再画建筑footprint
    py5.stroke_weight(0.5 / scale)
    for geom in buildings_geoms:
        for p in all_polygons(geom):
            draw_polygon(p, is_building=True)

    py5.pop_matrix()

    # 显示当前block信息
    py5.fill(0)
    py5.text_size(16)
    py5.text(f"Block: {block_id} ({current_index + 1}/{len(blocks)})", 10, 20)

def setup():
    py5.size(900, 900)
    draw_current_block()

def draw():
    pass  # 静态绘制

def key_pressed():
    global current_index
    if py5.key == '1':
        current_index = max(0, current_index - 1)
        draw_current_block()
    elif py5.key == '2':
        current_index = min(len(blocks) - 1, current_index + 1)
        draw_current_block()

py5.run_sketch()


E:\0_python_envs\urbanCode\Lib\site-packages\geopandas\io\sql.py:185: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


随机选取了 100 个block
